In [157]:
# install deps
!pip install playwright
!playwright install chromium

In [158]:
from playwright.async_api import async_playwright

VIDEO_URL = "https://www.youtube.com/watch?v=cdKop6aixVE" # up to 4k, no ads
EXPERIMENT_DURATION = 10 # how long the video should play in the experiment (in seconds)

In [159]:
playwright = await async_playwright().start()

browser = await playwright.chromium.launch(
    headless=False,
    args=[
        "--autoplay-policy=no-user-gesture-required",
        "--incognito",
    ],
)

# incognito context with cache disabled
context = await browser.new_context(
    viewport={"width": 1512, "height": 982},  # macbook 14 size
)
await context.set_extra_http_headers({"Cache-Control": "no-cache"})

page = await context.new_page()
print("Browser ready (incognito, cache disabled)")

Browser ready (incognito, cache disabled)


In [160]:
await page.goto(VIDEO_URL, wait_until="domcontentloaded")
await page.wait_for_timeout(3000)

# accept cookies
try:
    await page.click('button:has-text("Accept all")', timeout=5000)
    await page.wait_for_timeout(1000)
except Exception:
    pass

# pause the video
try:
    await page.click('button.ytp-play-button', timeout=5000)
except Exception:
    pass

await page.wait_for_timeout(3000)
print("Video loaded")

Video loaded


In [161]:
# set quality

await page.mouse.move(960, 540)
await page.wait_for_timeout(1000)

await page.locator('button.ytp-settings-button').hover()
await page.wait_for_timeout(500)
await page.locator('button.ytp-settings-button').click()
await page.wait_for_timeout(1000)

await page.click('div.ytp-menuitem-label:has-text("Quality")')
await page.wait_for_timeout(1000)

await page.click('div.ytp-menuitem-label:has-text("720")')
await page.wait_for_timeout(1000)

print("Quality set to 720p")

Quality set to 720p


In [162]:
# enter fullscreen
await page.keyboard.press('f')
await page.wait_for_timeout(1000)

# click play
await page.locator('button.ytp-play-button').click()
await page.wait_for_timeout(3000)

print("Playing... -> Ready to start measuring!!")

Playing... -> Ready to start measuring!!


In [163]:
# here we should start measuring
await page.wait_for_timeout(EXPERIMENT_DURATION * 1000)

print("Done.")

Done.


In [164]:
await browser.close()
await playwright.stop()

print("Browser closed")

Browser closed
